# Week 2 — The model is just a rule you can read

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abdulwahab-git/week-01-Assignment/blob/main/notebooks/02_your_first_readable_model.ipynb?flush_cache=true)

You'll:
1. Write a **1-line hand rule** and rank pages with it.
2. Fit a **depth-2 decision tree** and `print` it — the model "learned" a readable if/else. Then compare — where does it beat your rule, and where doesn’t it?
3. See **why you never feed the outcome back in** — that's leakage.

The payoff isn't a high score. It's: *my intuition was rough, the model found the real signal, and I can read exactly what it found.*

## 0. Setup (Colab or local)

In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    # find the repo root from wherever this kernel started
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

import pandas as pd, numpy as np
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# The label: a page is 'declining' when its recent trend is down. Simple, honest starter label.
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)
print(df.shape[0], "pages |  declining rate:", round(df["is_declining_label"].mean(), 3))

30000 pages |  declining rate: 0.542


## 1. A rule you write by hand: `stale x visible`
Intuition: a page worth reviewing is one that is **stale** (not updated in a while) **and** still **visible** (getting impressions). Rank those by how much exposure they have.

In [2]:
stale   = (df["days_since_last_update"] >= 180).astype(int)
visible = (df["impressions_90d"] >= 500).astype(int)
df["hand_rule_score"] = stale * visible * df["impressions_90d"]

top10 = df.sort_values("hand_rule_score", ascending=False).head(10)
top10[["impressions_90d", "days_since_last_update", "avg_position", "ctr", "trend_direction"]]

,impressions_90d,days_since_last_update,avg_position,ctr,trend_direction
16751,61678,194,19.7,0.15,down
16514,59472,194,24.8,0.13,down
7021,25715,194,22.2,0.23,down
21268,13299,193,10.5,0.49,down
11489,7812,194,39.0,0.01,down
12045,7558,193,17.9,0.20,down
698,4590,194,31.0,0.00,down
5327,4556,194,16.4,0.33,down
26810,4429,194,25.3,0.38,down
20837,1697,193,15.8,0.12,down


We need a way to score any ranking. **Precision@K** = of the top K pages a ranking flags, what fraction are actually declining.

In [3]:
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    topk = np.asarray(labels)[order[:k]]
    return topk.mean()

y = df["is_declining_label"].values
for k in (20, 50):
    print(f"Hand rule  Precision@{k}: {precision_at_k(df['hand_rule_score'], y, k):.3f}")

Hand rule  Precision@20: 0.900
Hand rule  Precision@50: 0.680


## 2. Let a model learn the rule — then read it
A **depth-2 decision tree** can only ask 3 yes/no questions. That constraint is the point: whatever it learns, you can read.

We give it a few **pre-decision** signals — never product flags.

In [4]:
from sklearn.tree import DecisionTreeClassifier, export_text

features = ["content_age_days", "days_since_last_update", "impressions_90d",
            "avg_position", "ctr", "word_count"]
X = df[features].replace([np.inf, -np.inf], np.nan).fillna(0)

tree = DecisionTreeClassifier(max_depth=2, class_weight="balanced", random_state=42)
tree.fit(X, y)

print(export_text(tree, feature_names=features))

|--- impressions_90d <= 5.50
|   |--- avg_position <= 0.75
|   |   |--- class: 0
|   |--- avg_position >  0.75
|   |   |--- class: 0
|--- impressions_90d >  5.50
|   |--- content_age_days <= 312.50
|   |   |--- class: 1
|   |--- content_age_days >  312.50
|   |   |--- class: 0



That printout **is** the model — a human-readable if/else. Now rank pages by the tree's probability and score it the same way.

In [5]:
tree_score = tree.predict_proba(X)[:, 1]
for k in (20, 50):
    hr = precision_at_k(df["hand_rule_score"], y, k)
    tr = precision_at_k(tree_score, y, k)
    print(f"Precision@{k}:  hand rule {hr:.3f}   vs   tree {tr:.3f}")

Precision@20:  hand rule 0.900   vs   tree 0.550
Precision@50:  hand rule 0.680   vs   tree 0.600


Look closely: the tree **wins at Precision@50** but your hand rule **wins at Precision@20**. Both results are real. A sharp human rule can be excellent at the very top of the list; the model's advantage shows up deeper, where simple rules run out of signal. Saying exactly that — instead of "the model is better" — is what honest evaluation sounds like.

## 3. Why you can't feed the outcome back in
Your label is `trend_direction == "down"`, and `trend_pct` is the exact percentage change that bucket is computed from — so it **is** the answer in disguise. Watch what happens if you feed it in as a feature:

In [6]:
X_leaky = df[features + ["trend_pct"]].replace([np.inf, -np.inf], np.nan).fillna(0)
leaky = DecisionTreeClassifier(max_depth=2, class_weight="balanced", random_state=42).fit(X_leaky, y)
print(f"'Leaky' tree Precision@50: {precision_at_k(leaky.predict_proba(X_leaky)[:,1], y, 50):.3f}  <- looks amazing")
print(export_text(leaky, feature_names=features + ["trend_pct"]))

'Leaky' tree Precision@50: 1.000  <- looks amazing
|--- trend_pct <= -20.05
|   |--- word_count <= 212.00
|   |   |--- class: 1
|   |--- word_count >  212.00
|   |   |--- class: 1
|--- trend_pct >  -20.05
|   |--- trend_pct <= -19.95
|   |   |--- class: 0
|   |--- trend_pct >  -19.95
|   |   |--- class: 0



The tree just split on `trend_pct` and nailed the label — because the label is **derived from** `trend_pct`. That's **leakage**: the feature is the answer in disguise, and it teaches you nothing.

That's also why the starter data ships **only observable signals** — the product's own decision flags (health scores, "needs CTR fix", and so on) aren't included, so you can't accidentally train on them. You build from what was knowable *before* the outcome.

> Rule of thumb: if a feature would only be known *because someone already made the decision you're predicting*, it leaks. Leave it out.

## 4. 🔧 Your turn
- Change `max_depth` to 3 or 4 — does Precision@50 improve? Can you still read the tree?
- Swap in different features (drop `impressions_90d`, add `engagement_rate`). What does the tree choose to split on first?
- **Important caveat:** we scored *in-sample* here for teaching. The real pipeline uses **client-holdout** validation (`scripts/03_train_model.py`) so a client's pages never appear in both train and test. Re-run your comparison with a train/test split and see if the gap holds.

Write your experiment below.

In [10]:
from sklearn.tree import DecisionTreeClassifier, export_text

# --- Experiment with max_depth = 3 ---
print("\n--- Experimenting with max_depth = 3 ---")
tree_depth3 = DecisionTreeClassifier(max_depth=3, class_weight="balanced", random_state=42)
tree_depth3.fit(X, y)

tree_score_depth3 = tree_depth3.predict_proba(X)[:, 1]
precision_depth3 = precision_at_k(tree_score_depth3, y, 50)
print(f"Decision Tree (max_depth=3) Precision@50: {precision_depth3:.3f}")
print("\nTree structure (max_depth=3):\n")
print(export_text(tree_depth3, feature_names=features))



--- Experimenting with max_depth = 3 ---
Decision Tree (max_depth=3) Precision@50: 0.720

Tree structure (max_depth=3):

|--- impressions_90d <= 5.50
|   |--- avg_position <= 0.75
|   |   |--- impressions_90d <= 3.50
|   |   |   |--- class: 0
|   |   |--- impressions_90d >  3.50
|   |   |   |--- class: 0
|   |--- avg_position >  0.75
|   |   |--- content_age_days <= 108.50
|   |   |   |--- class: 0
|   |   |--- content_age_days >  108.50
|   |   |   |--- class: 0
|--- impressions_90d >  5.50
|   |--- content_age_days <= 312.50
|   |   |--- ctr <= 0.33
|   |   |   |--- class: 1
|   |   |--- ctr >  0.33
|   |   |   |--- class: 1
|   |--- content_age_days >  312.50
|   |   |--- avg_position <= 25.15
|   |   |   |--- class: 0
|   |   |--- avg_position >  25.15
|   |   |   |--- class: 0



### Experiment: Swapping Features (drop `impressions_90d`, add `engagement_rate`)
Let's see how the decision tree changes when we remove `impressions_90d` and include `engagement_rate`.

In [12]:
# Define the new set of features
new_features = ["content_age_days", "days_since_last_update",
                "avg_position", "ctr", "word_count", "engagement_rate"]

# Prepare the new feature matrix X
X_new = df[new_features].replace([np.inf, -np.inf], np.nan).fillna(0)

# Train a new Decision Tree with max_depth=3 on the new features
tree_new_features = DecisionTreeClassifier(max_depth=3, class_weight="balanced", random_state=42)
tree_new_features.fit(X_new, y)

print("\nTree structure with new features (max_depth=3):\n")
print(export_text(tree_new_features, feature_names=new_features))

# Calculate Precision@50 for the new tree
tree_new_score = tree_new_features.predict_proba(X_new)[:, 1]
precision_new_features = precision_at_k(tree_new_score, y, 50)
print(f"\nDecision Tree (new features, max_depth=3) Precision@50: {precision_new_features:.3f}")


Tree structure with new features (max_depth=3):

|--- avg_position <= 0.55
|   |--- avg_position <= 0.15
|   |   |--- word_count <= 669.50
|   |   |   |--- class: 0
|   |   |--- word_count >  669.50
|   |   |   |--- class: 0
|   |--- avg_position >  0.15
|   |   |--- days_since_last_update <= 62.00
|   |   |   |--- class: 0
|   |   |--- days_since_last_update >  62.00
|   |   |   |--- class: 1
|--- avg_position >  0.55
|   |--- content_age_days <= 287.50
|   |   |--- ctr <= 0.33
|   |   |   |--- class: 1
|   |   |--- ctr >  0.33
|   |   |   |--- class: 1
|   |--- content_age_days >  287.50
|   |   |--- avg_position <= 38.45
|   |   |   |--- class: 0
|   |   |--- avg_position >  38.45
|   |   |   |--- class: 0


Decision Tree (new features, max_depth=3) Precision@50: 0.740


### Re-running Comparison with Train/Test Split

To get a more realistic evaluation, we'll split our data into training and testing sets. We'll train our models on the training data and evaluate their performance on the unseen test data. This helps assess how well the models generalize.

In [15]:
from sklearn.model_selection import train_test_split

# Prepare features and labels (using original features for initial comparison)
# Assuming X, y, and df are already defined from previous cells
# new_features = ["content_age_days", "days_since_last_update", "avg_position", "ctr", "word_count", "engagement_rate"]

# Split data into training and testing sets (80% train, 20% test)
X_train, X_test, y_train, y_test, df_train, df_test = train_test_split(X, y, df, test_size=0.2, random_state=42, stratify=y)

print(f"Training set size: {len(X_train)} samples")
print(f"Test set size: {len(X_test)} samples")

# --- 1. Re-evaluate Hand Rule on Test Set ---
hand_rule_precision_test_20 = precision_at_k(df_test["hand_rule_score"], y_test, 20)
hand_rule_precision_test_50 = precision_at_k(df_test["hand_rule_score"], y_test, 50)
print(f"\nHand Rule Precision@20 (Test): {hand_rule_precision_test_20:.3f}")
print(f"Hand Rule Precision@50 (Test): {hand_rule_precision_test_50:.3f}")


# --- 3. Re-train and evaluate Decision Tree (max_depth=3) on Test Set ---
tree_d3_test = DecisionTreeClassifier(max_depth=3, class_weight="balanced", random_state=42)
tree_d3_test.fit(X_train, y_train)
tree_d3_score_test = tree_d3_test.predict_proba(X_test)[:, 1]
tree_d3_precision_test_20 = precision_at_k(tree_d3_score_test, y_test, 20)
tree_d3_precision_test_50 = precision_at_k(tree_d3_score_test, y_test, 50)
print(f"\nDecision Tree (max_depth=3) Precision@20 (Test): {tree_d3_precision_test_20:.3f}")
print(f"Decision Tree (max_depth=3) Precision@50 (Test): {tree_d3_precision_test_50:.3f}")

# --- 4. Re-train and evaluate Decision Tree (max_depth=3, new features) on Test Set ---
# Need to prepare X_new_train and X_new_test based on the new_features list
X_new_full = df[new_features].replace([np.inf, -np.inf], np.nan).fillna(0)
X_new_train, X_new_test, _, _ = train_test_split(X_new_full, y, test_size=0.2, random_state=42, stratify=y)

tree_new_features_d3_test = DecisionTreeClassifier(max_depth=3, class_weight="balanced", random_state=42)
tree_new_features_d3_test.fit(X_new_train, y_train)
tree_new_features_d3_score_test = tree_new_features_d3_test.predict_proba(X_new_test)[:, 1]
tree_new_features_d3_precision_test_20 = precision_at_k(tree_new_features_d3_score_test, y_test, 20)
tree_new_features_d3_precision_test_50 = precision_at_k(tree_new_features_d3_score_test, y_test, 50)
print(f"\nDecision Tree (new features, max_depth=3) Precision@20 (Test): {tree_new_features_d3_precision_test_20:.3f}")
print(f"Decision Tree (new features, max_depth=3) Precision@50 (Test): {tree_new_features_d3_precision_test_50:.3f}")

# --- Compare all models on test set ---
print("\n--- Overall Comparison on Test Set ---")
for k in (20, 50):
    hr_test = precision_at_k(df_test["hand_rule_score"], y_test, k)
    tr3_test = precision_at_k(tree_d3_score_test, y_test, k)
    tr_new_f_test = precision_at_k(tree_new_features_d3_score_test, y_test, k)
    print(f"Precision@{k} (Test):  hand rule {hr_test:.3f} vs   tree(d3) {tr3_test:.3f}   vs   tree(d3_new_f) {tr_new_f_test:.3f}")


Training set size: 24000 samples
Test set size: 6000 samples

Hand Rule Precision@20 (Test): 0.650
Hand Rule Precision@50 (Test): 0.640

Decision Tree (max_depth=3) Precision@20 (Test): 0.650
Decision Tree (max_depth=3) Precision@50 (Test): 0.680

Decision Tree (new features, max_depth=3) Precision@20 (Test): 0.500
Decision Tree (new features, max_depth=3) Precision@50 (Test): 0.660

--- Overall Comparison on Test Set ---
Precision@20 (Test):  hand rule 0.650 vs   tree(d3) 0.650   vs   tree(d3_new_f) 0.500
Precision@50 (Test):  hand rule 0.640 vs   tree(d3) 0.680   vs   tree(d3_new_f) 0.660


### Save your work
**Colab:** *File → Save a copy in GitHub* (your submission) and *File → Save a copy in Drive*.

You now have the two core reflexes of applied ML: **discover before you model**, and **prefer a model you can read and can't fool**. That's the whole foundation the capstone builds on.